In [1]:
import json
import logging
import sqlite3
import sys
from pathlib import Path
from typing import Optional

from mcp.server.fastmcp import FastMCP
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
SERVER_DIR = PROJECT_ROOT / "mcp_servers" / "schema"
DATABASE_PATH = DATA_DIR / "verisql_demo.db"
SERVER_PATH = SERVER_DIR / "server.py"

DATA_DIR.mkdir(parents=True, exist_ok=True)
SERVER_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Database path:", DATABASE_PATH.resolve())
print("Server path:", SERVER_PATH.resolve())
print("Python executable:", sys.executable)
print("Environment setup successful.")

Project root: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL
Database path: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\data\verisql_demo.db
Server path: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\mcp_servers\schema\server.py
Python executable: C:\Users\monis\AppData\Local\Programs\Python\Python313\python.exe
Environment setup successful.


In [2]:
connection = sqlite3.connect(DATABASE_PATH)
connection.execute("PRAGMA foreign_keys = ON")

connection.executescript("""
CREATE TABLE IF NOT EXISTS departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL,
    location TEXT
);

CREATE TABLE IF NOT EXISTS employees (
    employee_id INTEGER PRIMARY KEY,
    employee_name TEXT NOT NULL,
    email TEXT,
    salary REAL,
    department_id INTEGER,
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

CREATE TABLE IF NOT EXISTS projects (
    project_id INTEGER PRIMARY KEY,
    project_name TEXT NOT NULL,
    budget REAL
);

CREATE TABLE IF NOT EXISTS employee_projects (
    employee_id INTEGER,
    project_id INTEGER,
    assigned_date TEXT,
    PRIMARY KEY (employee_id, project_id),
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id),
    FOREIGN KEY (project_id) REFERENCES projects(project_id)
);
""")

connection.executemany("INSERT OR IGNORE INTO departments VALUES (?, ?, ?)", [
    (1, "Engineering", "Bangalore"),
    (2, "Analytics", "Hyderabad"),
    (3, "Human Resources", "Delhi")
])

connection.executemany("INSERT OR IGNORE INTO employees VALUES (?, ?, ?, ?, ?)", [
    (1, "Arun", "arun@example.com", 75000, 1),
    (2, "Priya", "priya@example.com", 82000, 2),
    (3, "Rahul", "rahul@example.com", 68000, 1),
    (4, "Sneha", "sneha@example.com", 90000, 3),
    (5, "Kiran", "kiran@example.com", 78000, 2)
])

connection.executemany("INSERT OR IGNORE INTO projects VALUES (?, ?, ?)", [
    (1, "AI Platform", 500000),
    (2, "Data Warehouse", 300000),
    (3, "HR Automation", 150000)
])

connection.executemany("INSERT OR IGNORE INTO employee_projects VALUES (?, ?, ?)", [
    (1, 1, "2026-01-10"),
    (2, 2, "2026-02-15"),
    (3, 1, "2026-03-01"),
    (4, 3, "2026-03-12"),
    (5, 2, "2026-04-05")
])

connection.commit()
connection.close()

print("Database created:", DATABASE_PATH.exists())
print("Database location:", DATABASE_PATH.resolve())

Database created: True
Database location: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\data\verisql_demo.db


In [3]:
connection = sqlite3.connect(DATABASE_PATH)
tables = connection.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name").fetchall()

for (table,) in tables:
    print(f"\nTable: {table}")
    print("Columns:", connection.execute(f'PRAGMA table_info("{table}")').fetchall())
    print("Foreign keys:", connection.execute(f'PRAGMA foreign_key_list("{table}")').fetchall())

connection.close()
print("\nTotal tables:", len(tables))


Table: departments
Columns: [(0, 'department_id', 'INTEGER', 0, None, 1), (1, 'department_name', 'TEXT', 1, None, 0), (2, 'location', 'TEXT', 0, None, 0)]
Foreign keys: []

Table: employee_projects
Columns: [(0, 'employee_id', 'INTEGER', 0, None, 1), (1, 'project_id', 'INTEGER', 0, None, 2), (2, 'assigned_date', 'TEXT', 0, None, 0)]
Foreign keys: [(0, 0, 'projects', 'project_id', 'project_id', 'NO ACTION', 'NO ACTION', 'NONE'), (1, 0, 'employees', 'employee_id', 'employee_id', 'NO ACTION', 'NO ACTION', 'NONE')]

Table: employees
Columns: [(0, 'employee_id', 'INTEGER', 0, None, 1), (1, 'employee_name', 'TEXT', 1, None, 0), (2, 'email', 'TEXT', 0, None, 0), (3, 'salary', 'REAL', 0, None, 0), (4, 'department_id', 'INTEGER', 0, None, 0)]
Foreign keys: [(0, 0, 'departments', 'department_id', 'department_id', 'NO ACTION', 'NO ACTION', 'NONE')]

Table: projects
Columns: [(0, 'project_id', 'INTEGER', 0, None, 1), (1, 'project_name', 'TEXT', 1, None, 0), (2, 'budget', 'REAL', 0, None, 0)]
Fore

In [4]:
server_code = '''
import sqlite3
from pathlib import Path
from typing import Optional
from mcp.server.fastmcp import FastMCP

mcp_server = FastMCP("VeriSQL-Schema-MCP")

DATABASES = {"verisql_demo": Path(__file__).resolve().parents[2] / "data" / "verisql_demo.db"}
SENSITIVE_COLUMNS = {"email", "phone", "password", "token", "ssn", "credit_card"}

def get_connection(db_id: str):
    if db_id not in DATABASES:
        raise ValueError("Unauthorised database.")

    path = DATABASES[db_id].resolve()
    if not path.is_file():
        raise FileNotFoundError("Database file not found.")

    connection = sqlite3.connect(f"file:{path.as_posix()}?mode=ro", uri=True, timeout=5)
    connection.execute("PRAGMA query_only = ON")
    return connection

@mcp_server.tool()
def list_databases() -> list[str]:
    """List authorised databases."""
    return list(DATABASES.keys())

@mcp_server.tool()
def get_schema(db_id: str, tables: Optional[list[str]] = None) -> dict:
    """Retrieve table columns, primary keys and foreign keys."""
    connection = get_connection(db_id)
    try:
        available = {row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'")}
        selected = sorted(available if tables is None else set(tables))
        if not set(selected).issubset(available):
            raise ValueError("Unknown table requested.")

        result = {}
        for table in selected:
            columns = connection.execute(f'PRAGMA table_info("{table}")').fetchall()
            foreign_keys = connection.execute(f'PRAGMA foreign_key_list("{table}")').fetchall()
            result[table] = {
                "columns": [{"name": c[1], "type": c[2], "primary_key": bool(c[5]), "nullable": not bool(c[3]), "sensitive": c[1].lower() in SENSITIVE_COLUMNS} for c in columns],
                "foreign_keys": [{"column": f[3], "referenced_table": f[2], "referenced_column": f[4]} for f in foreign_keys]
            }
        return result
    finally:
        connection.close()

@mcp_server.tool()
def sample_values(db_id: str, table: str, column: str, n: int = 5) -> list:
    """Retrieve limited distinct values from non-sensitive columns."""
    if not 1 <= n <= 10:
        raise ValueError("n must be between 1 and 10.")

    schema = get_schema(db_id, [table])
    columns = {c["name"] for c in schema[table]["columns"]}
    if column not in columns:
        raise ValueError("Unknown column.")
    if column.lower() in SENSITIVE_COLUMNS:
        raise ValueError("Sensitive column samples are restricted.")

    connection = get_connection(db_id)
    try:
        query = f'SELECT DISTINCT "{column}" FROM "{table}" WHERE "{column}" IS NOT NULL LIMIT ?'
        return [row[0] for row in connection.execute(query, (n,)).fetchall()]
    finally:
        connection.close()

@mcp_server.tool()
def find_relevant_columns(db_id: str, question: str, k: int = 10) -> list[dict]:
    """Find relevant columns using basic keyword matching."""
    if not question.strip() or len(question) > 1000:
        raise ValueError("Invalid question.")
    if not 1 <= k <= 30:
        raise ValueError("k must be between 1 and 30.")

    schema = get_schema(db_id)
    terms = set(question.lower().replace("?", "").replace(",", "").split())
    matches = []

    for table, details in schema.items():
        for column in details["columns"]:
            name = column["name"]
            if name.lower() in SENSITIVE_COLUMNS:
                continue
            score = int(name.lower() in terms) * 3 + int(table.lower() in terms) * 2
            if score:
                matches.append({"table": table, "column": name, "score": score})

    return sorted(matches, key=lambda item: item["score"], reverse=True)[:k]

if __name__ == "__main__":
    mcp_server.run(transport="stdio")
'''

SERVER_PATH.write_text(server_code, encoding="utf-8")
compile(server_code, str(SERVER_PATH), "exec")

print("Schema MCP server created successfully.")
print("Syntax validation: PASSED")
print("Server path:", SERVER_PATH.resolve())

Schema MCP server created successfully.
Syntax validation: PASSED
Server path: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\mcp_servers\schema\server.py


In [5]:
server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)], env=None)

async def test_schema_mcp():
    with open(PROJECT_ROOT / "schema_mcp_errors.log", "w", encoding="utf-8") as server_log:
        async with stdio_client(server_params, errlog=server_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                print("Discovered tools:", [tool.name for tool in tools.tools])
                databases = await session.call_tool("list_databases", {})
                print("Available databases:", databases.content)
                schema = await session.call_tool("get_schema", {"db_id": "verisql_demo"})
                print("Retrieved schema:", schema.content)

await test_schema_mcp()

Discovered tools: ['list_databases', 'get_schema', 'sample_values', 'find_relevant_columns']
Available databases: [TextContent(type='text', text='verisql_demo', annotations=None, meta=None)]
Retrieved schema: [TextContent(type='text', text='{\n  "departments": {\n    "columns": [\n      {\n        "name": "department_id",\n        "type": "INTEGER",\n        "primary_key": true,\n        "nullable": true,\n        "sensitive": false\n      },\n      {\n        "name": "department_name",\n        "type": "TEXT",\n        "primary_key": false,\n        "nullable": false,\n        "sensitive": false\n      },\n      {\n        "name": "location",\n        "type": "TEXT",\n        "primary_key": false,\n        "nullable": true,\n        "sensitive": false\n      }\n    ],\n    "foreign_keys": []\n  },\n  "employee_projects": {\n    "columns": [\n      {\n        "name": "employee_id",\n        "type": "INTEGER",\n        "primary_key": true,\n        "nullable": true,\n        "sensitive"

In [6]:
async def test_sample_values():
    with open(PROJECT_ROOT / "schema_mcp_errors.log", "w", encoding="utf-8") as server_log:
        async with stdio_client(server_params, errlog=server_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                result = await session.call_tool("sample_values", {"db_id": "verisql_demo", "table": "employees", "column": "employee_name", "n": 5})
                print("Employee names:", result.content)

                result = await session.call_tool("sample_values", {"db_id": "verisql_demo", "table": "employees", "column": "email", "n": 5})
                print("Email access blocked:", result.isError)
                print("Response:", result.content)

await test_sample_values()

Employee names: [TextContent(type='text', text='Arun', annotations=None, meta=None), TextContent(type='text', text='Priya', annotations=None, meta=None), TextContent(type='text', text='Rahul', annotations=None, meta=None), TextContent(type='text', text='Sneha', annotations=None, meta=None), TextContent(type='text', text='Kiran', annotations=None, meta=None)]
Email access blocked: True
Response: [TextContent(type='text', text='Error executing tool sample_values: Sensitive column samples are restricted.', annotations=None, meta=None)]


In [7]:
async def test_relevant_columns():
    with open(PROJECT_ROOT / "schema_mcp_errors.log", "w", encoding="utf-8") as server_log:
        async with stdio_client(server_params, errlog=server_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool("find_relevant_columns", {"db_id": "verisql_demo", "question": "Find employees working in the Engineering department", "k": 10})
                print("Relevant columns:", result.content)

await test_relevant_columns()

Relevant columns: [TextContent(type='text', text='{\n  "table": "employees",\n  "column": "employee_id",\n  "score": 2\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employees",\n  "column": "employee_name",\n  "score": 2\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employees",\n  "column": "salary",\n  "score": 2\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employees",\n  "column": "department_id",\n  "score": 2\n}', annotations=None, meta=None)]


In [8]:
server_text = SERVER_PATH.read_text(encoding="utf-8")
start = server_text.index("def find_relevant_columns")
end = server_text.index('\nif __name__ == "__main__":', start)

new_function = '''
@mcp_server.tool()
def find_relevant_columns(db_id: str, question: str, k: int = 10) -> list[dict]:
    """Find relevant columns using column names and database values."""
    if not question.strip() or len(question) > 1000:
        raise ValueError("Invalid question.")
    if not 1 <= k <= 30:
        raise ValueError("k must be between 1 and 30.")

    schema = get_schema(db_id)
    normalize = lambda value: " ".join("".join(c.lower() if c.isalnum() else " " for c in str(value)).split())
    question_text = normalize(question)
    terms = set(question_text.split())
    matches = []
    connection = get_connection(db_id)

    try:
        for table, details in schema.items():
            for column in details["columns"]:
                name = column["name"]
                if column["sensitive"]:
                    continue

                column_terms = set(normalize(name.replace("_", " ")).split())
                score = len(column_terms & terms) * 3

                quoted_table = '"' + table.replace('"', '""') + '"'
                quoted_column = '"' + name.replace('"', '""') + '"'
                query = f"SELECT DISTINCT {quoted_column} FROM {quoted_table} WHERE {quoted_column} IS NOT NULL LIMIT 100"

                for row in connection.execute(query).fetchall():
                    value = normalize(row[0])
                    if value and value in question_text:
                        score += 5
                        break

                if score > 0:
                    matches.append({"table": table, "column": name, "score": score})
    finally:
        connection.close()

    return sorted(matches, key=lambda item: (-item["score"], item["table"], item["column"]))[:k]
'''

SERVER_PATH.write_text(server_text[:start] + new_function.strip() + server_text[end:], encoding="utf-8")
compile(SERVER_PATH.read_text(encoding="utf-8"), str(SERVER_PATH), "exec")

print("Relevance retrieval updated.")
print("Syntax validation: PASSED")
print("MCP startup block preserved:", 'mcp_server.run(transport="stdio")' in SERVER_PATH.read_text(encoding="utf-8"))

Relevance retrieval updated.
Syntax validation: PASSED
MCP startup block preserved: True


In [9]:
await test_relevant_columns()

Relevant columns: [TextContent(type='text', text='{\n  "table": "departments",\n  "column": "department_name",\n  "score": 8\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "departments",\n  "column": "department_id",\n  "score": 3\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employees",\n  "column": "department_id",\n  "score": 3\n}', annotations=None, meta=None)]


In [10]:
server_text = SERVER_PATH.read_text(encoding="utf-8")
start = server_text.index("@mcp_server.tool()\ndef find_relevant_columns")
end = server_text.index('\nif __name__ == "__main__":', start)

new_function = '''
@mcp_server.tool()
def find_relevant_columns(db_id: str, question: str, k: int = 10) -> list[dict]:
    """Find relevant columns using names, table context and database values."""
    if not question.strip() or len(question) > 1000:
        raise ValueError("Invalid question.")
    if not 1 <= k <= 30:
        raise ValueError("k must be between 1 and 30.")

    def normalize(value):
        words = "".join(c.lower() if c.isalnum() else " " for c in str(value)).split()
        return {word[:-1] if len(word) > 3 and word.endswith("s") else word for word in words}

    schema = get_schema(db_id)
    terms = normalize(question)
    matches = []
    connection = get_connection(db_id)

    try:
        for table, details in schema.items():
            table_score = len(normalize(table) & terms) * 2

            for column in details["columns"]:
                name = column["name"]
                if column["sensitive"]:
                    continue

                score = table_score + len(normalize(name.replace("_", " ")) & terms) * 3
                quoted_table = '"' + table.replace('"', '""') + '"'
                quoted_column = '"' + name.replace('"', '""') + '"'
                query = f"SELECT DISTINCT {quoted_column} FROM {quoted_table} WHERE {quoted_column} IS NOT NULL LIMIT 100"

                for row in connection.execute(query).fetchall():
                    value = normalize(row[0])
                    if value and value.issubset(terms):
                        score += 5
                        break

                if score > 2:
                    matches.append({"table": table, "column": name, "score": score})
    finally:
        connection.close()

    return sorted(matches, key=lambda item: (-item["score"], item["table"], item["column"]))[:k]
'''

SERVER_PATH.write_text(server_text[:start] + new_function.strip() + server_text[end:], encoding="utf-8")
compile(SERVER_PATH.read_text(encoding="utf-8"), str(SERVER_PATH), "exec")

print("Updated successfully.")
print("Syntax validation: PASSED")

Updated successfully.
Syntax validation: PASSED


In [11]:
server_text = SERVER_PATH.read_text(encoding="utf-8")
start = server_text.index("def find_relevant_columns")
end = server_text.index('\nif __name__ == "__main__":', start)

print("SERVER FILE:", SERVER_PATH.resolve())
print(server_text[start:end])

SERVER FILE: C:\Users\monis\OneDrive\Desktop\AI\Projects\SQL\mcp_servers\schema\server.py
def find_relevant_columns(db_id: str, question: str, k: int = 10) -> list[dict]:
    """Find relevant columns using names, table context and database values."""
    if not question.strip() or len(question) > 1000:
        raise ValueError("Invalid question.")
    if not 1 <= k <= 30:
        raise ValueError("k must be between 1 and 30.")

    def normalize(value):
        words = "".join(c.lower() if c.isalnum() else " " for c in str(value)).split()
        return {word[:-1] if len(word) > 3 and word.endswith("s") else word for word in words}

    schema = get_schema(db_id)
    terms = normalize(question)
    matches = []
    connection = get_connection(db_id)

    try:
        for table, details in schema.items():
            table_score = len(normalize(table) & terms) * 2

            for column in details["columns"]:
                name = column["name"]
                if column["sensitive"]:

In [12]:
server_text = SERVER_PATH.read_text(encoding="utf-8")
start = server_text.index("def find_relevant_columns")
print(server_text[start - 150:start + 150])

 return [row[0] for row in connection.execute(query, (n,)).fetchall()]
    finally:
        connection.close()

@mcp_server.tool()
@mcp_server.tool()
def find_relevant_columns(db_id: str, question: str, k: int = 10) -> list[dict]:
    """Find relevant columns using names, table context and database 


In [13]:
server_text = SERVER_PATH.read_text(encoding="utf-8")
server_text = server_text.replace("@mcp_server.tool()\n@mcp_server.tool()\ndef find_relevant_columns", "@mcp_server.tool()\ndef find_relevant_columns")

compile(server_text, str(SERVER_PATH), "exec")
SERVER_PATH.write_text(server_text, encoding="utf-8")

print("Duplicate decorator removed.")
print("Syntax validation: PASSED")

Duplicate decorator removed.
Syntax validation: PASSED


In [14]:
await test_relevant_columns()

Relevant columns: [TextContent(type='text', text='{\n  "table": "departments",\n  "column": "department_name",\n  "score": 10\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "departments",\n  "column": "department_id",\n  "score": 5\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employee_projects",\n  "column": "employee_id",\n  "score": 5\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employees",\n  "column": "department_id",\n  "score": 5\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employees",\n  "column": "employee_id",\n  "score": 5\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employees",\n  "column": "employee_name",\n  "score": 5\n}', annotations=None, meta=None)]


In [15]:
async def schema_dummy_agent(question: str, db_id: str = "verisql_demo"):
    with open(PROJECT_ROOT / "schema_mcp_errors.log", "w", encoding="utf-8") as server_log:
        async with stdio_client(server_params, errlog=server_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()

                relevant = await session.call_tool("find_relevant_columns", {"db_id": db_id, "question": question, "k": 10})
                schema = await session.call_tool("get_schema", {"db_id": db_id})

                if relevant.isError or schema.isError:
                    raise RuntimeError("Schema MCP retrieval failed.")

                result = {"question": question, "relevant_columns": relevant.content, "schema": schema.content}
                print("QUESTION:", question)
                print("\nRELEVANT COLUMNS:", relevant.content)
                print("\nSCHEMA RETRIEVED SUCCESSFULLY.")

                return result

In [16]:
result = await schema_dummy_agent("Find the names of employees working in the Engineering department")

QUESTION: Find the names of employees working in the Engineering department

RELEVANT COLUMNS: [TextContent(type='text', text='{\n  "table": "departments",\n  "column": "department_name",\n  "score": 13\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employees",\n  "column": "employee_name",\n  "score": 8\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "departments",\n  "column": "department_id",\n  "score": 5\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employee_projects",\n  "column": "employee_id",\n  "score": 5\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employees",\n  "column": "department_id",\n  "score": 5\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "employees",\n  "column": "employee_id",\n  "score": 5\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "table": "projects",\n  "column": "

In [17]:
async def test_guardrails():
    tests = [
        ("Invalid database", "get_schema", {"db_id": "unauthorised_db"}),
        ("Unknown column", "sample_values", {"db_id": "verisql_demo", "table": "employees", "column": "unknown_column"}),
        ("Invalid limit", "find_relevant_columns", {"db_id": "verisql_demo", "question": "Find employees", "k": 0}),
        ("Sensitive email", "sample_values", {"db_id": "verisql_demo", "table": "employees", "column": "email"})
    ]

    with open(PROJECT_ROOT / "schema_mcp_errors.log", "w", encoding="utf-8") as server_log:
        async with stdio_client(server_params, errlog=server_log) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                for name, tool, args in tests:
                    response = await session.call_tool(tool, args)
                    print(f"{name}: {'PASSED' if response.isError else 'FAILED'}")

await test_guardrails()

Invalid database: PASSED
Unknown column: PASSED
Invalid limit: PASSED
Sensitive email: PASSED


In [18]:
connection = sqlite3.connect(f"file:{DATABASE_PATH.resolve().as_posix()}?mode=ro", uri=True)

try:
    connection.execute("INSERT INTO departments (department_id, department_name) VALUES (99, 'Test')")
    print("FAILED: Write operation was allowed.")
except sqlite3.OperationalError as error:
    print("PASSED: Write operation blocked.")
    print("Reason:", error)
finally:
    connection.close()

PASSED: Write operation blocked.
Reason: attempt to write a readonly database
